# La meilleure droite

| Module | Durée | Prérequis |
|---|---|---|
| M6 – Orthogonalité et moindres carrés | 75 min | M5 |

In [ ]:
# Style graphique Liouaï : couleurs du logo, axes sobres
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

LIOUAI = {"navy": "#243743", "vert": "#28B78D", "vert_texte": "#167A5C", "gris": "#A1A9AE",
          "gris_texte": "#5C6870", "jaune": "#E5C800", "fond": "#F3F5F6"}
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[LIOUAI["vert"], LIOUAI["navy"], LIOUAI["gris"],
                                         LIOUAI["jaune"], LIOUAI["vert_texte"], LIOUAI["gris_texte"]]),
    "axes.edgecolor": LIOUAI["gris"], "axes.labelcolor": LIOUAI["navy"],
    "axes.titlecolor": LIOUAI["navy"], "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": LIOUAI["gris_texte"], "ytick.color": LIOUAI["gris_texte"],
    "text.color": LIOUAI["navy"], "grid.color": "#E1E6E9", "legend.frameon": False,
    "font.family": "sans-serif", "font.sans-serif": ["Roboto", "Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 100,
})
# Dégradé de marque : gris clair -> vert -> navy (et sa version inversée)
if "liouai" not in mpl.colormaps:
    _cmap = LinearSegmentedColormap.from_list("liouai", [LIOUAI["fond"], LIOUAI["vert"], LIOUAI["navy"]])
    mpl.colormaps.register(_cmap)
    mpl.colormaps.register(_cmap.reversed())

## ① Définir la question

Le club de physique étudie le rebond d'une balle de caoutchouc. Deux groupes, A et B, ont lâché la balle de différentes hauteurs, deux fois par hauteur, et ont filmé le rebond pour mesurer la hauteur atteinte. Les mesures sont dans `donnees/rebonds.csv`.

La théorie prévoit que la balle remonte toujours à la **même fraction** de sa hauteur de départ : $r = k\,h$. Le coefficient $k$ caractérise la balle (et le sol).

**Les questions du club : quelle est cette fraction $k$ ? Les mesures confirment-elles le modèle ? Peut-on faire confiance à toutes les mesures ?**

**Activité** : Les points ne seront jamais exactement alignés. Pourquoi ? Faites la liste des sources d'erreur possibles, et classez-les : erreurs **aléatoires** (qui changent d'un essai à l'autre) ou **systématiques** (qui faussent toujours dans le même sens).

Indiquez vos réponses et justifiez-les.

## ② Estimer avant de calculer

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rebonds = pd.read_csv("donnees/rebonds.csv")
h = rebonds["hauteur_lacher_cm"].to_numpy(dtype=float)
r = rebonds["hauteur_rebond_cm"].to_numpy(dtype=float)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(h, r, "o", color=LIOUAI["navy"])
ax.set_xlabel("hauteur de lâcher h (cm)"); ax.set_ylabel("hauteur du rebond r (cm)"); ax.grid(alpha=0.3)
ax.set_xlim(0, 200); ax.set_ylim(0, 130); ax.set_title("Les 44 mesures du club")
plt.tight_layout(); plt.show()

**Activité** : Avec une règle posée sur le graphique, tracez « à l'œil » la droite qui vous semble la meilleure. Lisez sa pente.

In [ ]:
pari_pente = 0.5        # fraction de la hauteur récupérée au rebond (modifiez)

## ③ Abstraire : un système sans solution

On cherche une droite $r = a\,h + b$ (on laisse une ordonnée à l'origine $b$, pour vérifier que la théorie, qui prévoit $b = 0$, est confirmée). Chaque mesure donne une équation :

$$
\begin{pmatrix} h_1 & 1 \\ h_2 & 1 \\ \vdots & \vdots \\ h_{44} & 1 \end{pmatrix}
\begin{pmatrix} a \\ b \end{pmatrix}
\approx
\begin{pmatrix} r_1 \\ r_2 \\ \vdots \\ r_{44} \end{pmatrix}
\qquad\text{soit}\qquad A\mathbf{x} \approx \mathbf{r}.
$$

44 équations, 2 inconnues : aucune droite ne passe par tous les points. On choisit celle qui rend **la somme des carrés des résidus** la plus petite.

![Schéma : les résidus sont les écarts verticaux entre les points mesurés et la droite ; on minimise la somme de leurs carrés](../../assets/schemas/schema-residus.svg)

**Le lien avec le notebook 6.1.** La somme des carrés des résidus est le carré de la distance $\lVert \mathbf{r} - A\mathbf{x} \rVert^2$ entre le vecteur des mesures et un point du plan des colonnes de $A$. La minimiser, c'est **projeter** $\mathbf{r}$ sur ce plan : on résout les équations normales

$$
A^{\mathsf{T}}A\,\hat{\mathbf{x}} = A^{\mathsf{T}}\mathbf{r}.
$$

**Activité** : Pour les trois points $(1\,;\,1)$, $(2\,;\,3)$, $(3\,;\,4)$, écrivez $A$, calculez $A^{\mathsf{T}}A$ et $A^{\mathsf{T}}\mathbf{r}$, puis résolvez le système $2 \times 2$. Vous devez trouver $a = 1{,}5$ et $b = -1/3$.

Indiquez vos réponses et justifiez-les.

## ④ Calculer

### Trois manières d'obtenir la même droite

In [ ]:
A = np.column_stack([h, np.ones_like(h)])

x_normales = np.linalg.solve(A.T @ A, A.T @ r)
x_lstsq, *_ = np.linalg.lstsq(A, r, rcond=None)
x_polyfit = np.polyfit(h, r, 1)                 # renvoie aussi (pente, ordonnée à l'origine)

for nom, (a_, b_) in [("équations normales", x_normales), ("np.linalg.lstsq", x_lstsq), ("np.polyfit", x_polyfit)]:
    print(f"{nom:20s} : r = {a_:.4f} h + {b_:.3f}")

### Regarder les résidus

Une droite qui « passe bien » ne suffit pas : il faut regarder les **résidus**, c'est-à-dire ce que le modèle n'explique pas. S'ils sont dus au hasard, ils doivent ressembler à un nuage sans forme.

In [ ]:
a, b = x_lstsq
residus = r - (a * h + b)
groupe_B = (rebonds["groupe"] == "B").to_numpy()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
ax = axes[0]
hh = np.linspace(0, 190, 2)
ax.plot(h[~groupe_B], r[~groupe_B], "o", color=LIOUAI["navy"], label="groupe A")
ax.plot(h[groupe_B], r[groupe_B], "^", color=LIOUAI["vert"], label="groupe B")
ax.plot(hh, a * hh + b, color=LIOUAI["jaune"], lw=2, label=f"r = {a:.3f} h + {b:.2f}")
ax.set_xlabel("h (cm)"); ax.set_ylabel("r (cm)"); ax.legend(); ax.grid(alpha=0.3); ax.set_title("La droite des moindres carrés")
ax = axes[1]
ax.axhline(0, color=LIOUAI["gris"])
ax.plot(h[~groupe_B], residus[~groupe_B], "o", color=LIOUAI["navy"], label="groupe A")
ax.plot(h[groupe_B], residus[groupe_B], "^", color=LIOUAI["vert"], label="groupe B")
ax.set_xlabel("h (cm)"); ax.set_ylabel("résidu (cm)"); ax.legend(); ax.grid(alpha=0.3); ax.set_title("Les résidus")
plt.tight_layout(); plt.show()

print(f"Résidu moyen du groupe A : {residus[~groupe_B].mean():+.2f} cm")
print(f"Résidu moyen du groupe B : {residus[groupe_B].mean():+.2f} cm")
k = np.argmax(np.abs(residus))
print(f"Plus grand résidu : {residus[k]:+.1f} cm (groupe {rebonds['groupe'][k]}, h = {h[k]:.0f} cm, essai {rebonds['essai'][k]})")

**Activité** : Les résidus révèlent deux anomalies. Décrivez-les. Laquelle est une erreur **systématique**, laquelle est une erreur **ponctuelle** ? Proposez une explication physique pour chacune.

Indiquez vos réponses et justifiez-les.

### Intégrer l'erreur systématique au modèle

Le groupe B a mesuré la hauteur du rebond au **sommet** de la balle, et non à sa base : toutes ses mesures sont décalées d'une même quantité $d$. On peut l'estimer en ajoutant une colonne au modèle, qui vaut 1 pour les mesures du groupe B et 0 sinon :

$$
r = a\,h + b + d \times [\text{groupe B}] .
$$

Le modèle reste **linéaire en ses paramètres** $a$, $b$, $d$ : les moindres carrés s'appliquent tels quels. On retire aussi la mesure aberrante, après avoir vérifié qu'elle s'explique (la balle a touché le mètre ruban, d'après le cahier du groupe A).

In [ ]:
garde = np.ones(len(r), dtype=bool)
garde[k] = False
A3 = np.column_stack([h, np.ones_like(h), groupe_B.astype(float)])
(a3, b3, d3), *_ = np.linalg.lstsq(A3[garde], r[garde], rcond=None)
print(f"r = {a3:.4f} h + {b3:.2f}   et décalage du groupe B : d = {d3:.2f} cm")

residus3 = r[garde] - A3[garde] @ np.array([a3, b3, d3])
print(f"Écart type des résidus : {residus.std():.2f} cm avant, {residus3.std():.2f} cm après")

### Et sans ordonnée à l'origine ?

La théorie dit $r = k\,h$. Avec une seule colonne $\mathbf{h}$, la meilleure droite passant par l'origine est la projection du notebook 6.1 :

$$
\hat{k} = \frac{\mathbf{h} \cdot \mathbf{r}}{\mathbf{h} \cdot \mathbf{h}} .
$$

In [ ]:
r_corrige = r - d3 * groupe_B            # on retire le décalage estimé du groupe B
k_chapeau = (h[garde] @ r_corrige[garde]) / (h[garde] @ h[garde])
print(f"k = {k_chapeau:.4f} : la balle remonte à {k_chapeau:.0%} de sa hauteur")
print(f"Coefficient de restitution e = racine(k) = {np.sqrt(k_chapeau):.3f}")

## ⑤ Interpréter

In [ ]:
print(f"Votre pente estimée à l'œil : {pari_pente} ; pente des moindres carrés : {k_chapeau:.3f}")

**Activité** :

1. Énoncez le résultat pour le club en une phrase, avec une précision raisonnable (combien de chiffres ?).
2. L'ordonnée à l'origine $b$ trouvée est-elle compatible avec la théorie $b = 0$ ? Qu'est-ce qui permettrait d'en juger ?
3. Que pensez-vous de la droite tracée à l'œil ? A-t-elle été plus influencée par la mesure aberrante, ou moins ?
4. Regardez à nouveau les résidus après correction : ont-ils encore une forme ? Que pourrait-elle signifier physiquement (pensez au frottement de l'air pour les grandes hauteurs) ?

Indiquez vos réponses et justifiez-les.

## ⑥ Critiquer et généraliser

- **Une hypothèse cachée.** Les moindres carrés mesurent l'erreur **verticalement**, sur $r$ seulement. On a donc supposé que $h$ est mesurée sans erreur. Est-ce raisonnable ici ?
- **Sensibilité.** Un seul point aberrant a-t-il beaucoup déplacé la droite ? Comparez les pentes avec et sans lui. Pourquoi le **carré** des résidus donne-t-il beaucoup de poids aux grands écarts ?
- **Généraliser.** Écrivez une fonction `influence(h, r)` qui, pour chaque mesure, calcule la pente obtenue **sans** cette mesure. Tracez ces pentes : quelles mesures influencent le plus le résultat ?

In [ ]:
def influence(h, r):
    # À compléter : renvoie un tableau des pentes obtenues en retirant chaque mesure à tour de rôle
    ...

## Corrigé

*À n'ouvrir qu'après avoir cherché.*

In [ ]:
def influence(h, r):
    pentes = []
    for i in range(len(h)):
        masque = np.arange(len(h)) != i
        A_i = np.column_stack([h[masque], np.ones(masque.sum())])
        pentes.append(np.linalg.lstsq(A_i, r[masque], rcond=None)[0][0])
    return np.array(pentes)

pentes = influence(h, r)
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.axhline(a, color=LIOUAI["gris"], label="pente avec toutes les mesures")
ax.plot(pentes, "o", color=LIOUAI["navy"])
ax.set_xlabel("mesure retirée (numéro de ligne)"); ax.set_ylabel("pente obtenue"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
i = np.argmax(np.abs(pentes - a))
print(f"La mesure la plus influente est la ligne {i} : sans elle, la pente passe de {a:.4f} à {pentes[i]:.4f}")

## Bilan — auto-évaluation

Pour chaque compétence travaillée dans ce notebook, choisissez un niveau : **0** non observé · **1** guidé · **2** autonome · **3** transfert.
La cellule enregistre votre bilan dans `progression.json`, à la racine du site.

In [ ]:
import json, datetime, pathlib

bilan = {
    "Estimer la solution avant de calculer": 1,
    "Énoncer le résultat dans le contexte du problème": 1,
    "Identifier, énoncer et justifier les hypothèses": 1,
    "Identifier erreurs systématiques et aléatoires": 1,
    "Repérer les traits intéressants du résultat": 1,
}   # modifiez les niveaux

ici = pathlib.Path.cwd()
racine = next((p for p in [ici, *ici.parents] if (p / "myst.yml").exists()), ici)
fichier = racine / "progression.json"
historique = json.loads(fichier.read_text(encoding="utf-8")) if fichier.exists() else []
historique.append({"notebook": "6.2-la-meilleure-droite", "date": datetime.date.today().isoformat(), "niveaux": bilan})
fichier.write_text(json.dumps(historique, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Bilan enregistré ({len(historique)} entrée(s) dans {fichier.name}).")